<a href="https://colab.research.google.com/github/BF667-IDLE/rvc-batch/blob/main/rvc_batch_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RVC-Batch: Voice Conversion Demo

A simple, high-quality voice conversion tool focused on ease of use and batch processing.

**Features:**
- Single file & batch folder inference
- Multiple F0 methods (RMVPE, CREPE, FCPE, SWIPE, etc.)
- RVC v1 & v2 model support
- CUDA / MPS / CPU support

## 1. Installation

Install dependencies and clone the repository.

In [ ]:
#@title Clone repo & install dependencies
!git clone https://github.com/BF667-IDLE/rvc-batch.git
%cd rvc-batch
!uv pip install -r requirements.txt

## 2. Download Models

Download the HuBERT base model and your preferred F0 predictor.

In [ ]:
#@title Download required models
import os
os.makedirs("models", exist_ok=True)

from main.utils import check_predictors, check_embedders

f0_method = "rmvpe"  # @param ["rmvpe", "crepe-full", "crepe-tiny", "crepe-small", "crepe-medium", "crepe-large", "fcpe", "swipe"]

print(f"Downloading HuBERT model...")
check_embedders()

print(f"Downloading {f0_method} predictor...")
check_predictors(f0_method)

print("All models downloaded!")

## 3. Load Your Voice Model

Upload your RVC voice model (`.pth`) and optionally an index file (`.index`).
Or provide a Google Drive / direct download link.

In [ ]:

#@title Download voice model from URL
import os
import zipfile
import glob
from google.colab import files

model_source = "url"  # @param ["upload", "url"]

os.makedirs("models", exist_ok=True)

if model_source == "upload":
    print("Upload your .pth voice model file:")
    uploaded = files.upload()
    model_file = list(uploaded.keys())[0]
    os.rename(model_file, f"models/{model_file}")
    voice_model_path = f"models/{model_file}"

    print("\nUpload your .index file (optional, click Cancel to skip):")
    try:
        uploaded_idx = files.upload()
        idx_file = list(uploaded_idx.keys())[0]
        os.rename(idx_file, f"models/{idx_file}")
        index_path = f"models/{idx_file}"
        print(f"Index saved to {index_path}")
    except:
        index_path = ""
        print("No index file provided.")
else:
    model_url = "https://huggingface.co/Dr-Ivo-Robotnik/DaveV3/resolve/main/DaveV3.zip?download=true"  # @param {type:"string"}

    # Download the ZIP file
    zip_filename = os.path.basename(model_url).split('?')[0]
    zip_path = f"models/{zip_filename}"

    print(f"Downloading model from {model_url}...")
    if not os.path.exists(zip_path):
        !wget -O "{zip_path}" "{model_url}"
    else:
        print(f"Using existing file: {zip_filename}")

    # Extract the ZIP file
    print(f"Extracting {zip_filename}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall("models/")

    # Find the model folder (assuming it's named after the model)
    extracted_folders = [f for f in os.listdir("models/")
                        if os.path.isdir(os.path.join("models/", f)) and f != "__MACOSX"]

    if extracted_folders:
        # Get the most recently created folder
        model_folder = max(extracted_folders,
                          key=lambda f: os.path.getctime(os.path.join("models/", f)))
        model_folder_path = os.path.join("models/", model_folder)

        print(f"Found model folder: {model_folder}")

        # Find .pth file in the model folder
        pth_files = glob.glob(os.path.join(model_folder_path, "*.pth"))
        if pth_files:
            voice_model_path = pth_files[0]
            print(f"✅ Found model file: {os.path.basename(voice_model_path)}")
        else:
            # If no .pth in subfolder, check the extracted root
            pth_files = glob.glob("models/*.pth")
            if pth_files:
                voice_model_path = pth_files[0]
                print(f"✅ Found model file: {os.path.basename(voice_model_path)}")
            else:
                raise Exception("❌ No .pth file found in the extracted archive")

        # Find .index file in the model folder
        index_files = glob.glob(os.path.join(model_folder_path, "*.index"))
        if index_files:
            index_path = index_files[0]
            print(f"✅ Found index file: {os.path.basename(index_path)}")
        else:
            index_path = ""
            print("ℹ️ No index file found in the archive")
    else:
        # If no folders were extracted, look for files directly in models directory
        pth_files = glob.glob("models/*.pth")
        if pth_files:
            voice_model_path = pth_files[0]
            print(f"✅ Found model file: {os.path.basename(voice_model_path)}")
        else:
            raise Exception("❌ No .pth file found in the extracted archive")

        # Look for index file
        index_files = glob.glob("models/*.index")
        if index_files:
            index_path = index_files[0]
            print(f"✅ Found index file: {os.path.basename(index_path)}")
        else:
            index_path = ""
            print("ℹ️ No index file found in the archive")

    # Clean up zip file (optional)
    # os.remove(zip_path)
    # print(f"Cleaned up {zip_filename}")

print(f"\n📁 Voice model: {voice_model_path}")
print(f"📁 Index file: {index_path if index_path else '(none)'}")

# Verify files exist
if os.path.exists(voice_model_path):
    print(f"✅ Model file ready")
else:
    print(f"⚠️ Warning: Model file not found")

if index_path and os.path.exists(index_path):
    print(f"✅ Index file ready")

In [ ]:
#@title Initialize models
import torch
import os, sys


from main.infer.infer import Config, load_hubert, get_vc

# Auto-detect device
if torch.cuda.is_available():
    device = "cuda:0"
    is_half = True
    print(f"GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = "mps"
    is_half = True
    print("Using Apple Silicon (MPS)")
else:
    device = "cpu"
    is_half = True
    print("Using CPU")

config = Config(device, is_half)

print("Loading HuBERT model...")
hubert_model = load_hubert(device, is_half, "models/hubert_base.pt")

print("Loading voice model...")
cpt, version, net_g, tgt_sr, vc = get_vc(device, is_half, config, voice_model_path)

print(f"\nModel version: {version}")
print(f"Target sample rate: {tgt_sr}Hz")
print("Ready for inference!")

## 4. Single File Inference

Convert a single audio file with full parameter control.

In [ ]:
#@title Single file inference
import os
from google.colab import files
from infer.infer import rvc_infer

# Upload input audio
print("Upload audio file to convert:")
uploaded = files.upload()
input_file = list(uploaded.keys())[0]

# Parameters
output_name = "output.wav"  # @param {type:"string"}
pitch_change = 0  # @param {type:"slider", min:-12, max:12, step:1}
f0_method = "rmvpe"  # @param ["rmvpe", "crepe-full", "crepe-tiny", "crepe-small", "crepe-medium", "crepe-large", "fcpe", "swipe", "pm", "harvest", "dio", "yin", "pyin"]
index_rate = 0.75  # @param {type:"slider", min:0, max:1, step:0.05}
filter_radius = 3  # @param {type:"integer"}
rms_mix_rate = 0.25  # @param {type:"slider", min:0, max:1, step:0.05}
protect = 0.33  # @param {type:"slider", min:0, max:0.5, step:0.01}

os.makedirs("output", exist_ok=True)
output_path = f"output/{output_name}"

print(f"\nConverting {input_file}...")
print(f"  Pitch: {pitch_change:+d} st | F0: {f0_method} | Index rate: {index_rate}")

rvc_infer(
    index_path=index_path,
    index_rate=index_rate,
    input_path=input_file,
    output_path=output_path,
    pitch_change=pitch_change,
    f0_method=f0_method,
    cpt=cpt,
    version=version,
    net_g=net_g,
    filter_radius=filter_radius,
    tgt_sr=tgt_sr,
    rms_mix_rate=rms_mix_rate,
    protect=protect,
    crepe_hop_length=128,
    vc=vc,
    hubert_model=hubert_model,
)

print(f"\nDone! Output saved to {output_path}")
files.download(output_path)

## 5. Batch Folder Inference

Process an entire folder of audio files at once.
Supports: `.wav`, `.mp3`, `.flac`, `.ogg`, `.opus`, `.m4a`, `.wma`, `.aac`, `.webm`

In [ ]:
#@title Batch folder inference
import os
import shutil
from google.colab import files
from infer.infer import rvc_infer_batch

# Upload a ZIP file or multiple audio files
print("Upload audio files or a ZIP archive:")
uploaded = files.upload()

batch_input_dir = "input_batch"
batch_output_dir = "output_batch"
os.makedirs(batch_input_dir, exist_ok=True)
os.makedirs(batch_output_dir, exist_ok=True)

for fname, data in uploaded.items():
    fpath = os.path.join(batch_input_dir, fname)
    with open(fpath, "wb") as f:
        f.write(data)
    print(f"  Saved: {fname}")

# Extract ZIP if uploaded
for fname in os.listdir(batch_input_dir):
    if fname.endswith(".zip"):
        print(f"\nExtracting {fname}...")
        shutil.unpack_archive(os.path.join(batch_input_dir, fname), batch_input_dir)
        os.remove(os.path.join(batch_input_dir, fname))

# Parameters
pitch_change = 0  # @param {type:"slider", min:-12, max:12, step:1}
f0_method = "rmvpe"  # @param ["rmvpe", "crepe-full", "crepe-tiny", "crepe-small", "crepe-medium", "crepe-large", "fcpe", "swipe", "pm", "harvest", "dio", "yin", "pyin"]
index_rate = 0.75  # @param {type:"slider", min:0, max:1, step:0.05}
filter_radius = 3  # @param {type:"integer"}
rms_mix_rate = 0.25  # @param {type:"slider", min:0, max:1, step:0.05}
protect = 0.33  # @param {type:"slider", min:0, max:0.5, step:0.01}

# Count files
audio_extensions = {".wav", ".mp3", ".flac", ".ogg", ".opus", ".m4a", ".wma", ".aac", ".webm"}
file_count = sum(1 for f in os.listdir(batch_input_dir) if os.path.splitext(f)[1].lower() in audio_extensions)

print(f"\n{'='*50}")
print(f"Batch inference: {file_count} file(s)")
print(f"  Pitch: {pitch_change:+d} st | F0: {f0_method} | Index rate: {index_rate}")
print(f"{'='*50}")

summary = rvc_infer_batch(
    index_path=index_path,
    index_rate=index_rate,
    input_path=batch_input_dir,
    output_path=batch_output_dir,
    pitch_change=pitch_change,
    f0_method=f0_method,
    cpt=cpt,
    version=version,
    net_g=net_g,
    filter_radius=filter_radius,
    tgt_sr=tgt_sr,
    rms_mix_rate=rms_mix_rate,
    protect=protect,
    crepe_hop_length=128,
    vc=vc,
    hubert_model=hubert_model,
)

print(f"\nResults: {summary['processed']} processed, {summary['failed']} failed, {summary['skipped']} skipped")

# Download all results as ZIP
zip_path = "output_batch.zip"
shutil.make_archive("output_batch", "zip", batch_output_dir)
files.download(zip_path)

## 6. Playback Results

Listen to converted files directly in Colab.

In [ ]:
#@title Listen to output files
import glob
from IPython.display import Audio, display

output_dir = "output"  # @param ["output", "output_batch"]

wav_files = sorted(glob.glob(os.path.join(output_dir, "*.wav")))

if not wav_files:
    print(f"No .wav files found in {output_dir}/")
else:
    print(f"Found {len(wav_files)} file(s):\n")
    for f in wav_files:
        print(f"  {os.path.basename(f)}")
        display(Audio(filename=f))